# Phase 4: Integrated CoT Applications
## Intelligent Technique Selection + Domain-Specific Solvers

Combine all CoT techniques from Phases 1-3 into a unified intelligent system.
Then apply to two real-world domains: **Financial Analysis** and **AP Calculus**.

## Section 1: Setup and Imports

In [ ]:
import os
import json
import re
import time
from typing import Dict, List, Tuple
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

print('✅ Imports successful')

## Section 2: Load Local Model

In [ ]:
# Load local Mistral-7B model
from transformers import pipeline
import torch

print("Loading Mistral-7B model (local inference)...")

try:
    device = 0 if torch.cuda.is_available() else -1
    device_str = "GPU" if device == 0 else "CPU"
    print(f"Using: {device_str}")
    
    generator = pipeline(
        "text-generation",
        model="mistralai/Mistral-7B-Instruct-v0.1",
        torch_dtype=torch.float16 if device == 0 else torch.float32,
        device_map="auto" if device == 0 else None,
        device=device
    )
    
    print("✅ Mistral-7B model loaded!")
    
    def call_model(prompt: str, temperature: float = 0.0, max_tokens: int = 300) -> str:
        """Generate text using local Mistral-7B model"""
        try:
            use_sampling = temperature > 0.0
            actual_temp = max(temperature, 0.1) if not use_sampling else temperature
            
            response = generator(
                prompt,
                max_new_tokens=max_tokens,
                temperature=actual_temp,
                num_return_sequences=1,
                do_sample=use_sampling
            )
            return response[0]["generated_text"]
        except Exception as e:
            raise e
    
    print("✅ Model configured")
    
except Exception as e:
    print(f"Error: {e}")
    print("Fallback: Using GPT-2")
    generator = pipeline("text-generation", model="gpt2", device=-1)
    
    def call_model(prompt: str, temperature: float = 0.0, max_tokens: int = 300) -> str:
        try:
            use_sampling = temperature > 0.0
            actual_temp = max(temperature, 0.1) if not use_sampling else temperature
            
            response = generator(
                prompt,
                max_new_tokens=max_tokens,
                temperature=actual_temp,
                num_return_sequences=1,
                do_sample=use_sampling
            )
            return response[0]["generated_text"]
        except Exception as e:
            raise e

## Section 3: Phase 4 Overview

### What is Phase 4?

Phases 1-3 taught you the **techniques**. Phase 4 shows you how to **apply** them to real problems.

In this phase, you'll:

1. **Build an Intelligent Router** - Automatically chooses between CoT, Self-Consistency, and Least-to-Most
2. **Solve Financial Problems** - Portfolio recommendations and investment analysis
3. **Solve Calculus Problems** - Derivatives, optimization, step-by-step proofs
4. **Measure Performance** - Compare techniques across domains

### The Key Insight

Different problems need different techniques:

| Problem Type | Best Technique | Why? |
|--------------|----------------|------|
| Simple math | Basic CoT | No need for complexity |
| Complex multi-step | Least-to-Most | Breaking down helps |
| High-stakes decision | Self-Consistency | Need confidence |
| Financial calculation | Self-Consistency + Confidence | Accuracy critical |
| Calculus proof | Least-to-Most + Confidence | Need step-by-step clarity |

### Architecture

```
IntelligentCoTRouter
├─ Analyzes problem (complexity, domain, stakes)
├─ Selects best technique
└─ Routes to domain-specific solver
   ├─ FinancialAnalyzer
   │  ├─ Portfolio Recommendations
   │  └─ Investment Comparisons
   └─ APCalculusTutor
      ├─ Derivative Problems
      └─ Optimization Problems
```

## Section 4: IntelligentCoTRouter Class

In [ ]:
class IntelligentCoTRouter:
    """Intelligently select and route between different CoT techniques"""
    
    def __init__(self, model_fn):
        self.model = model_fn
        self.technique_counts = Counter()
    
    def analyze_problem(self, problem: str) -> Dict:
        """Analyze problem to determine complexity and domain"""
        
        # Simple heuristics for complexity
        words = problem.split()
        complexity = len(words) // 20  # Rough proxy
        
        # Domain detection
        financial_keywords = ['stock', 'portfolio', 'investment', 'financial', 'return', 'yield', 'dividend']
        calculus_keywords = ['derivative', 'integral', 'optimization', 'maximize', 'minimize', 'calculus']
        
        is_financial = any(kw in problem.lower() for kw in financial_keywords)
        is_calculus = any(kw in problem.lower() for kw in calculus_keywords)
        
        # Stakeness detection
        critical_keywords = ['critical', 'important', 'essential', 'must', 'risk']
        is_critical = any(kw in problem.lower() for kw in critical_keywords)
        
        return {
            'complexity': min(complexity, 3),  # 0=simple, 1=medium, 2=complex, 3+=very complex
            'is_financial': is_financial,
            'is_calculus': is_calculus,
            'is_critical': is_critical,
            'word_count': len(words)
        }
    
    def route_problem(self, problem: str) -> str:
        """Determine best technique based on problem characteristics"""
        
        analysis = self.analyze_problem(problem)
        
        # Decision logic
        if analysis['is_critical']:
            technique = 'self-consistency'  # High stakes = need voting
        elif analysis['complexity'] >= 2:
            technique = 'least-to-most'  # Complex = decompose
        else:
            technique = 'basic-cot'  # Simple = just think step-by-step
        
        self.technique_counts[technique] += 1
        return technique
    
    def extract_answer(self, response: str) -> str:
        """Extract numerical answer from response"""
        # Try to find the final numerical answer
        numbers = re.findall(r'-?\d+\.?\d*', response)
        return numbers[-1] if numbers else "Unable to extract"
    
    def solve_basic_cot(self, problem: str) -> Dict:
        """Standard CoT: Show reasoning steps"""
        prompt = f"Let me think step by step.\n\n{problem}\n\nAnswer:"
        response = self.model(prompt, temperature=0.0, max_tokens=300)
        
        return {
            'method': 'basic-cot',
            'response': response,
            'answer': self.extract_answer(response),
            'confidence': 0.6  # No explicit confidence for basic CoT
        }
    
    def solve_least_to_most(self, problem: str) -> Dict:
        """Least-to-Most: Decompose and solve progressively"""
        
        # Step 1: Decompose
        decompose_prompt = f"Break this problem into simple steps:\n{problem}\n\nSteps:"
        decomposition = self.model(decompose_prompt, temperature=0.0, max_tokens=200)
        
        # Step 2: Solve progressively
        solve_prompt = f"Now solve step by step:\n{problem}\n\nSolution:"
        solution = self.model(solve_prompt, temperature=0.0, max_tokens=400)
        
        return {
            'method': 'least-to-most',
            'decomposition': decomposition,
            'response': solution,
            'answer': self.extract_answer(solution),
            'confidence': 0.75  # Higher confidence due to decomposition
        }
    
    def solve_self_consistency(self, problem: str, n_samples: int = 5) -> Dict:
        """Self-Consistency: Multiple reasoning paths with voting"""
        
        prompt = f"Let me think step by step.\n\n{problem}\n\nAnswer:"
        answers = []
        responses = []
        
        for i in range(n_samples):
            try:
                response = self.model(prompt, temperature=0.7, max_tokens=300)
                answers.append(self.extract_answer(response))
                responses.append(response)
            except Exception as e:
                pass
        
        # Vote on most common answer
        if answers:
            answer_counts = Counter(answers)
            final_answer = answer_counts.most_common(1)[0][0]
            confidence = answer_counts.most_common(1)[0][1] / len(answers)
        else:
            final_answer = "Error"
            confidence = 0
        
        return {
            'method': 'self-consistency',
            'n_samples': n_samples,
            'responses': responses,
            'answer': final_answer,
            'confidence': confidence,
            'all_answers': answers
        }
    
    def get_confidence(self, problem: str, answer: str) -> float:
        """Get model's confidence in the answer"""
        confidence_prompt = f"Problem: {problem}\nAnswer: {answer}\nConfidence (1-10):"
        response = self.model(confidence_prompt, temperature=0.0, max_tokens=50)
        
        numbers = re.findall(r'\d+', response)
        confidence = int(numbers[0]) if numbers else 5
        return min(confidence, 10) / 10.0
    
    def solve(self, problem: str) -> Dict:
        """Main solve method: analyze, route, and execute"""
        
        # Analyze problem
        analysis = self.analyze_problem(problem)
        
        # Route to best technique
        technique = self.route_problem(problem)
        
        # Execute appropriate technique
        if technique == 'self-consistency':
            result = self.solve_self_consistency(problem, n_samples=5)
        elif technique == 'least-to-most':
            result = self.solve_least_to_most(problem)
        else:
            result = self.solve_basic_cot(problem)
        
        # Add analysis metadata
        result['analysis'] = analysis
        result['technique_selected'] = technique
        
        return result

# Initialize router
router = IntelligentCoTRouter(call_model)
print("✅ IntelligentCoTRouter initialized")

## Section 5: FinancialAnalyzer Class

In [ ]:
class FinancialAnalyzer:
    """Domain-specific solver for financial analysis problems"""
    
    def __init__(self, router: IntelligentCoTRouter):
        self.router = router
        self.results = []
    
    def analyze_portfolio(self, portfolio: Dict) -> Dict:
        """Analyze investment portfolio"""
        
        problem = f"""
        Portfolio Analysis:
        Holdings: {portfolio}
        
        Analyze this portfolio and provide:
        1. Current allocation assessment
        2. Risk analysis
        3. Specific recommendations
        4. Expected returns estimate
        """
        
        result = self.router.solve(problem)
        result['problem_type'] = 'portfolio-analysis'
        self.results.append(result)
        return result
    
    def solve_financial_problem(self, problem: str) -> Dict:
        """Solve general financial problem with high confidence requirement"""
        
        # Ensure problem is marked as critical for routing
        critical_problem = f"CRITICAL FINANCIAL DECISION: {problem}"
        
        result = self.router.solve(critical_problem)
        result['problem_type'] = 'financial-calculation'
        self.results.append(result)
        return result
    
    def display_result(self, result: Dict) -> None:
        """Display financial analysis result"""
        
        print(f"\n{'='*80}")
        print(f"Financial Analysis Result")
        print(f"{'='*80}")
        print(f"\n📊 Method Used: {result['technique_selected'].upper()}")
        print(f"📈 Answer: {result['answer']}")
        print(f"💪 Confidence: {result['confidence']:.0%}")
        
        if 'decomposition' in result:
            print(f"\n📋 Decomposition:\n{result['decomposition'][:300]}...")
        
        if 'response' in result:
            print(f"\n💭 Reasoning:\n{result['response'][:400]}...")
        
        print(f"\n{'='*80}")

# Initialize financial analyzer
financial_analyzer = FinancialAnalyzer(router)
print("✅ FinancialAnalyzer initialized")

## Section 6: APCalculusTutor Class

In [ ]:
class APCalculusTutor:
    """Domain-specific solver for AP Calculus problems"""
    
    def __init__(self, router: IntelligentCoTRouter):
        self.router = router
        self.results = []
    
    def solve_derivative_problem(self, problem: str) -> Dict:
        """Solve derivative problem with step-by-step reasoning"""
        
        detailed_problem = f"""
        AP Calculus Derivative Problem:
        {problem}
        
        Show all steps clearly:
        1. Identify the rule needed (power, product, quotient, chain)
        2. Apply the rule carefully
        3. Simplify the result
        4. Show the final answer
        """
        
        result = self.router.solve(detailed_problem)
        result['problem_type'] = 'derivative'
        self.results.append(result)
        return result
    
    def solve_optimization_problem(self, problem: str) -> Dict:
        """Solve optimization (max/min) problem"""
        
        detailed_problem = f"""
        AP Calculus Optimization Problem:
        {problem}
        
        This requires finding max/min values. Solution process:
        1. Define all variables clearly
        2. Write the objective function to optimize
        3. Identify constraints
        4. Take the derivative and find critical points
        5. Use second derivative test or check endpoints
        6. State the maximum or minimum value
        7. Verify the answer is reasonable
        """
        
        result = self.router.solve(detailed_problem)
        result['problem_type'] = 'optimization'
        self.results.append(result)
        return result
    
    def solve_calculus_problem(self, problem: str) -> Dict:
        """General calculus problem solver"""
        
        detailed_problem = f"""
        AP Calculus Problem:
        {problem}
        
        Solve this step by step, showing all work.
        """
        
        result = self.router.solve(detailed_problem)
        result['problem_type'] = 'general-calculus'
        self.results.append(result)
        return result
    
    def display_result(self, result: Dict) -> None:
        """Display calculus solution result"""
        
        print(f"\n{'='*80}")
        print(f"AP Calculus Solution")
        print(f"{'='*80}")
        print(f"\n📐 Problem Type: {result['problem_type'].upper()}")
        print(f"🎯 Method Used: {result['technique_selected'].upper()}")
        print(f"✅ Answer: {result['answer']}")
        print(f"🔒 Confidence: {result['confidence']:.0%}")
        
        if 'decomposition' in result:
            print(f"\n📝 Decomposition:\n{result['decomposition'][:300]}...")
        
        if 'response' in result:
            print(f"\n💡 Solution:\n{result['response'][:400]}...")
        
        print(f"\n{'='*80}")

# Initialize calculus tutor
calculus_tutor = APCalculusTutor(router)
print("✅ APCalculusTutor initialized")

## Section 7: Financial Analysis Test Cases

In [ ]:
print("\n" + "="*80)
print("FINANCIAL ANALYSIS TEST CASES")
print("="*80)

# Test Case 1: Portfolio Recommendation
print("\n📊 Test 1: Portfolio Recommendation")
finanical_problem_1 = """
An investor has $100,000 to invest. They are risk-averse and want to maximize returns 
while minimizing volatility. Current market conditions show stocks at 3-year high, 
bonds offering 4% yield. What is the recommended allocation?
"""

financial_result_1 = financial_analyzer.solve_financial_problem(financial_problem_1)
financial_analyzer.display_result(financial_result_1)

# Test Case 2: Investment Comparison
print("\n💰 Test 2: Investment Comparison")
financial_problem_2 = """
Compare two investments:
Option A: Stock fund with 8% expected return, 12% volatility, $1000 minimum
Option B: Bond fund with 4% expected return, 3% volatility, $500 minimum

For a 5-year investment horizon with $5000 capital, which is better?
"""

financial_result_2 = financial_analyzer.solve_financial_problem(financial_problem_2)
financial_analyzer.display_result(financial_result_2)

## Section 8: AP Calculus Test Cases

In [ ]:
print("\n" + "="*80)
print("AP CALCULUS TEST CASES")
print("="*80)

# Test Case 1: Derivative Problem
print("\n📐 Test 1: Derivative Problem")
calculus_problem_1 = "Find the derivative of f(x) = 3x^3 - 2x^2 + 5x - 7"
calculus_result_1 = calculus_tutor.solve_derivative_problem(calculus_problem_1)
calculus_tutor.display_result(calculus_result_1)

# Test Case 2: Optimization Problem
print("\n🎯 Test 2: Optimization Problem")
calculus_problem_2 = """
A farmer needs to build a rectangular fence with one side against a barn (no fence needed).
They have 100 meters of fencing. What dimensions maximize the enclosed area?
"""
calculus_result_2 = calculus_tutor.solve_optimization_problem(calculus_problem_2)
calculus_tutor.display_result(calculus_result_2)

# Test Case 3: Chain Rule Problem
print("\n🔗 Test 3: Chain Rule Problem")
calculus_problem_3 = "Find the derivative of g(x) = (2x^2 + 3x)^5 using the chain rule"
calculus_result_3 = calculus_tutor.solve_calculus_problem(calculus_problem_3)
calculus_tutor.display_result(calculus_result_3)

## Section 9: Results Analysis and Comparison

In [ ]:
print("\n" + "="*80)
print("RESULTS ANALYSIS")
print("="*80)

# Technique usage statistics
technique_usage = dict(router.technique_counts)
print("\n📊 Technique Usage:")
for technique, count in technique_usage.items():
    percentage = (count / sum(technique_usage.values())) * 100
    print(f"  {technique}: {count} times ({percentage:.1f}%)")

# Confidence statistics
all_results = financial_analyzer.results + calculus_tutor.results
confidences = [r['confidence'] for r in all_results]

print(f"\n🎯 Confidence Metrics:")
print(f"  Average confidence: {np.mean(confidences):.2f}/1.0")
print(f"  Min confidence: {np.min(confidences):.2f}/1.0")
print(f"  Max confidence: {np.max(confidences):.2f}/1.0")

# Domain-specific results
financial_results = [r for r in all_results if 'financial' in r.get('problem_type', '')]
calculus_results = [r for r in all_results if 'calculus' in r.get('problem_type', '') or r.get('problem_type') in ['derivative', 'optimization', 'general-calculus']]

print(f"\n💰 Financial Problems: {len(financial_results)}")
if financial_results:
    print(f"  Average confidence: {np.mean([r['confidence'] for r in financial_results]):.2f}/1.0")
    finance_techniques = Counter([r['technique_selected'] for r in financial_results])
    for tech, count in finance_techniques.items():
        print(f"    {tech}: {count}")

print(f"\n📐 Calculus Problems: {len(calculus_results)}")
if calculus_results:
    print(f"  Average confidence: {np.mean([r['confidence'] for r in calculus_results]):.2f}/1.0")
    calc_techniques = Counter([r['technique_selected'] for r in calculus_results])
    for tech, count in calc_techniques.items():
        print(f"    {tech}: {count}")

# Create summary dataframe
summary_data = []
for result in all_results:
    summary_data.append({
        'Domain': 'Financial' if 'financial' in result.get('problem_type', '') else 'Calculus',
        'Problem Type': result.get('problem_type', 'unknown'),
        'Technique': result['technique_selected'],
        'Confidence': result['confidence'],
        'Answer': result['answer'][:20] + '...' if len(result['answer']) > 20 else result['answer']
    })

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print("\n📋 Summary Table:")
    print(summary_df.to_string(index=False))

## Section 10: Visualization

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Technique Usage
if technique_usage:
    ax = axes[0, 0]
    techniques = list(technique_usage.keys())
    counts = list(technique_usage.values())
    colors = ['#FF6B6B', '#4ECDC4', '#FFE66D']
    ax.bar(techniques, counts, color=colors[:len(techniques)], alpha=0.7, edgecolor='black', linewidth=2)
    ax.set_title('Technique Usage Across Problems', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

# Plot 2: Confidence Distribution
if confidences:
    ax = axes[0, 1]
    ax.hist(confidences, bins=10, color='#95E1D3', edgecolor='black', alpha=0.7)
    ax.set_title('Confidence Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel('Confidence Score', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.axvline(np.mean(confidences), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(confidences):.2f}')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

# Plot 3: Domain Comparison
if len(all_results) > 0:
    ax = axes[1, 0]
    domain_counts = Counter([r.get('problem_type', 'unknown').split('-')[0] for r in all_results])
    if domain_counts:
        domains = list(domain_counts.keys())
        counts = list(domain_counts.values())
        colors = ['#F38181', '#AA96DA']
        ax.barh(domains, counts, color=colors[:len(domains)], alpha=0.7, edgecolor='black', linewidth=2)
        ax.set_title('Problems by Domain', fontsize=12, fontweight='bold')
        ax.set_xlabel('Count', fontsize=10)
        ax.grid(axis='x', alpha=0.3)

# Plot 4: Technique-Confidence Correlation
if len(all_results) > 0:
    ax = axes[1, 1]
    technique_conf = {}
    for result in all_results:
        tech = result['technique_selected']
        if tech not in technique_conf:
            technique_conf[tech] = []
        technique_conf[tech].append(result['confidence'])
    
    techniques_plot = list(technique_conf.keys())
    avg_confs = [np.mean(technique_conf[t]) for t in techniques_plot]
    colors = ['#FF6B6B', '#4ECDC4', '#FFE66D']
    ax.bar(techniques_plot, avg_confs, color=colors[:len(techniques_plot)], alpha=0.7, edgecolor='black', linewidth=2)
    ax.set_title('Average Confidence by Technique', fontsize=12, fontweight='bold')
    ax.set_ylabel('Average Confidence', fontsize=10)
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('phase4_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Analysis visualization saved to phase4_analysis.png")

## Section 11: Save Results

In [ ]:
# Prepare results for JSON
results_to_save = {
    'timestamp': datetime.now().isoformat(),
    'total_problems_solved': len(all_results),
    'technique_usage': dict(router.technique_counts),
    'average_confidence': float(np.mean(confidences)) if confidences else 0,
    'financial_problems_solved': len(financial_results),
    'calculus_problems_solved': len(calculus_results),
    'problem_results': [
        {
            'problem_type': r.get('problem_type', 'unknown'),
            'technique_selected': r['technique_selected'],
            'confidence': float(r['confidence']),
            'answer': str(r['answer'])
        }
        for r in all_results
    ]
}

with open('phase4_results.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)

print("✅ Results saved to phase4_results.json")

# Also save as CSV
if summary_data:
    summary_df.to_csv('phase4_results.csv', index=False)
    print("✅ Results saved to phase4_results.csv")

print("\n" + "="*80)
print("🎉 PHASE 4 COMPLETE!")
print("="*80)
print("\nYou've successfully:")
print("✓ Built an Intelligent CoT Router")
print("✓ Created a Financial Analysis System")
print("✓ Created an AP Calculus Tutor")
print("✓ Applied all techniques from Phases 1-3")
print("✓ Analyzed results across domains")
print("\n🏆 Congratulations on completing the entire CoT deep dive!")